# Welcome to Week 2!

## Frontier Model APIs

In Week 1, we used multiple Frontier LLMs through their Chat UI, and we connected with the OpenAI's API.

Today we'll connect with them through their APIs..

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Important Note - Please read me</h2>
            <span style="color:#900;">I'm continually improving these labs, adding more examples and exercises.
            At the start of each week, it's worth checking you have the latest code.<br/>
            First do a git pull and merge your changes as needed</a>. Check out the GitHub guide for instructions. Any problems? Try asking ChatGPT to clarify how to merge - or contact me!<br/>
            </span>
        </td>
    </tr>
</table>
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder about the resources page</h2>
            <span style="color:#f71;">Here's a link to resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## Setting up your keys - OPTIONAL!

We're now going to try asking a bunch of models some questions!

This is totally optional. If you have keys to Anthropic, Gemini or others, then you can add them in.

If you'd rather not spend the extra, then just watch me do it!

For OpenAI, visit https://openai.com/api/  
For Anthropic, visit https://console.anthropic.com/  
For Google, visit https://ai.google.dev/gemini-api   
For DeepSeek, visit https://platform.deepseek.com/  
For Groq, visit https://console.groq.com/  
For Grok, visit https://console.x.ai/  


You can also use OpenRouter as your one-stop-shop for many of these! OpenRouter is "the unified interface for LLMs":

For OpenRouter, visit https://openrouter.ai/  


With each of the above, you typically have to navigate to:
1. Their billing page to add the minimum top-up (except Gemini, Groq, Google, OpenRouter may have free tiers)
2. Their API key page to collect your API key

### Adding API keys to your .env file

When you get your API keys, you need to set them as environment variables by adding them to your `.env` file.

```
OPENAI_API_KEY=xxxx
ANTHROPIC_API_KEY=xxxx
GOOGLE_API_KEY=xxxx
DEEPSEEK_API_KEY=xxxx
GROQ_API_KEY=xxxx
GROK_API_KEY=xxxx
OPENROUTER_API_KEY=xxxx
```

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Any time you change your .env file</h2>
            <span style="color:#900;">Remember to Save it! And also rerun load_dotenv(override=True)<br/>
            </span>
        </td>
    </tr>
</table>

In [13]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [33]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI
OpenRouter API Key exists and begins sk-


In [34]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()
# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

openai_model = "openai/gpt-4o"
gemini_model = "gemini-2.5-flash"

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
openai_url = "https://models.github.ai/inference"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
openai = OpenAI(api_key=openai_api_key, base_url=openai_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [15]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [16]:
response = openai.chat.completions.create(model=openai_model, messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

Sure, here's one for you:

Why did the aspiring LLM engineer bring a ladder to their code review?

Because they wanted to *scale* their model to new heights!

## Training vs Inference time scaling

In [17]:
easy_puzzle = [
    {"role": "user", "content": 
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

In [20]:
response = openai.chat.completions.create(model=openai_model, messages=easy_puzzle)
display(Markdown(response.choices[0].message.content))

0.5

In [25]:
#GPT-5 models can also pass on another parameter reasoning_effort 
response = gemini.chat.completions.create(model=gemini_model, messages=easy_puzzle)
display(Markdown(response.choices[0].message.content))

Let's denote the outcomes of the two coins as an ordered pair (Coin 1, Coin 2), where H is Heads and T is Tails.
The possible outcomes when tossing two coins are:
1. (H, H) - Both are heads
2. (H, T) - Coin 1 is heads, Coin 2 is tails
3. (T, H) - Coin 1 is tails, Coin 2 is heads
4. (T, T) - Both are tails

Each of these outcomes has a probability of 1/4, assuming the coins are fair and independent.

We are given the information: "One of them is heads."
This means that at least one of the coins shows heads. Let's call this event A.
The outcomes that satisfy event A are:
- (H, H) - At least one is heads (in fact, both are)
- (H, T) - At least one is heads
- (T, H) - At least one is heads
The outcome (T, T) does not satisfy this condition.

So, our reduced sample space, given event A, is { (H, H), (H, T), (T, H) }.
Each of these 3 outcomes is equally likely, so the probability of each in this reduced sample space is 1/3.

Now, we need to find the probability that "the other is tails" within this reduced sample space. Let's call this event B.
"The other is tails" means that if one coin is heads, the second coin must be tails. This implies that there is exactly one head and exactly one tail.

Let's examine the outcomes in our reduced sample space A:
- (H, H): If one coin is heads (e.g., the first one), the other coin (the second one) is also heads, not tails. So, (H, H) does NOT satisfy "the other is tails".
- (H, T): If one coin is heads (the first one), the other coin (the second one) IS tails. This outcome satisfies the condition.
- (T, H): If one coin is heads (the second one), the other coin (the first one) IS tails. This outcome also satisfies the condition.

So, the outcomes in the reduced sample space that satisfy "the other is tails" are { (H, T), (T, H) }.
There are 2 favorable outcomes.
There are 3 total possible outcomes in the reduced sample space.

Therefore, the probability is 2/3.

The final answer is $\boxed{\frac{2}{3}}$.

In [ ]:
response = openai.chat.completions.create(model="gpt-5-mini", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

BadRequestError: Error code: 400 - {'error': {'message': 'Unrecognized request argument supplied: reasoning_effort', 'type': 'invalid_request_error', 'param': None, 'code': None}}

## Testing out the best models on the planet

In [22]:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

In [23]:
response = openai.chat.completions.create(model=openai_model, messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))

Let's calculate the distance the worm gnawed through step by step.

### 1. **Understanding the scenario**
- There are two volumes of Pushkin on the bookshelf: Volume 1 and Volume 2.
- Each volume consists of:
  - **Pages**: 2 cm thick.
  - **Covers**: Each cover is 2 mm (0.2 cm) thick. Since each volume has two covers, the covers add up to \( 2 \times 0.2 = 0.4 \, \text{cm} \).
  - Total thickness of each volume = thickness of pages + thickness of covers:
    \[
    \text{Thickness of one volume} = 2 + 0.4 = 2.4 \, \text{cm}.
    \]

### 2. **Key point about the worm’s path**
The worm gnawed **from the first page of Volume 1 to the last page of Volume 2, perpendicular to the pages**.

On a bookshelf, books are typically positioned with the spines outward. Therefore:
- The **first page of Volume 1** is located on the **side opposite the spine**.
- The **last page of Volume 2** is located on the **side opposite the spine**.

Thus, the worm starts on the **outermost edge** of Volume 1 and gnaws through **both volumes completely**, reaching the **outermost edge** of Volume 2.

### 3. **Calculating the distance gnawed**
The worm gnaws through:
- The **pages** of Volume 1: \( 2 \, \text{cm} \).
- The **back cover** of Volume 1: \( 0.2 \, \text{cm} \).
- The **front cover** of Volume 2: \( 0.2 \, \text{cm} \).
- The **pages** of Volume 2: \( 2 \, \text{cm} \).

In total:
\[
\text{Total distance gnawed} = 2 \, \text{cm} + 0.2 \, \text{cm} + 0.2 \, \text{cm} + 2 \, \text{cm} = 4.4 \, \text{cm}.
\]

### Final Answer:
The worm gnawed through a total distance of **4.4 cm**.

In [ ]:
#response = anthropic.chat.completions.create(model="claude-sonnet-4-5-20250929", messages=hard_puzzle)
#display(Markdown(response.choices[0].message.content))

In [ ]:
response = openai.chat.completions.create(model="gpt-5", messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))

In [26]:
response = gemini.chat.completions.create(model=gemini_model, messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))

This is a classic riddle! The trick lies in how books are arranged on a shelf.

1.  **Visualize the books on the shelf:**
    *   Volume 1 is on the left.
    *   Volume 2 is on the right, next to Volume 1.
    *   The **spine** of each book faces outwards.
    *   For **Volume 1 (left book)**:
        *   Its **front cover** is on its **right side** (facing Volume 2).
        *   Its **back cover** is on its **left side** (facing the wall or end of the shelf).
        *   The "first page" is immediately *inside* its front cover.
    *   For **Volume 2 (right book)**:
        *   Its **front cover** is on its **right side** (facing the next book or end of the shelf).
        *   Its **back cover** is on its **left side** (facing Volume 1).
        *   The "last page" is immediately *inside* its back cover.

2.  **Trace the worm's path:**
    *   The worm starts "from the first page of the first volume." This means it starts *after* Volume 1's front cover. It does **not** gnaw through Volume 1's front cover.
    *   It then gnaws through **all the pages of the first volume**. (Thickness: 2 cm).
    *   It then gnaws through the **back cover of the first volume**. (Thickness: 2 mm).
    *   It then gnaws through the **front cover of the second volume**. (Thickness: 2 mm).
    *   It then reaches "the last page of the second volume." This means it stops *before* gnawing through any of the pages of the second volume (from the front) and before its back cover. It does **not** gnaw through Volume 2's pages or back cover.

3.  **Calculate the total distance:**
    *   Pages of Volume 1: 2 cm
    *   Back cover of Volume 1: 2 mm
    *   Front cover of Volume 2: 2 mm

    Convert all to the same unit (e.g., millimeters):
    *   2 cm = 20 mm

    Total distance = 20 mm + 2 mm + 2 mm = **24 mm**

    Or, in centimeters: **2.4 cm**

The worm gnawed through 24 mm (or 2.4 cm).

## A spicy challenge to test the competitive spirit

In [25]:
dilemma_prompt = """
You and a partner are contestants on a game show. You're each taken to separate rooms and given a choice:
Cooperate: Choose "Share" — if both of you choose this, you each win $1,000.
Defect: Choose "Steal" — if one steals and the other shares, the stealer gets $2,000 and the sharer gets nothing.
If both steal, you both get nothing.
Do you choose to Steal or Share? Pick one.
"""

dilemma = [
    {"role": "user", "content": dilemma_prompt},
]


In [26]:
response = openai.chat.completions.create(model=openai_model, messages=dilemma)
display(Markdown(response.choices[0].message.content))


This scenario mirrors the classic "Prisoner's Dilemma," a cornerstone of game theory. The choice depends on one's mindset — whether to prioritize trust and mutual gain (Share) or act selfishly to maximize individual payoff (Steal). Here's my reasoning:

I would choose **Share** because:

1. **Trust and Cooperation**: If both parties cooperate, the outcome is optimal: both win $1,000. This result is fair and promotes mutual trust.
   
2. **Avoiding Mutual Loss**: If both steal, no one benefits. Choosing "Share" is a hedge against this worst-case outcome.

3. **Ethical Consideration**: Acting in good faith aligns with values of fairness and integrity.

While "Steal" might be tempting as a way to maximize individual gain (if I assume my partner will choose "Share"), it relies on betrayal and potentially risks ending up with nothing if both parties defect. Choosing "Share" offers a path to collective gain and maintains trust, even in a competitive environment.



In [27]:
response = gemini.chat.completions.create(model=gemini_model, messages=dilemma)
display(Markdown(response.choices[0].message.content))

This is a classic game theory scenario, a variant of the Prisoner's Dilemma. Let's break down the outcomes from my perspective:

*   **If I choose SHARE:**
    *   If my partner also Shares: I get $1,000.
    *   If my partner Steals: I get $0.

*   **If I choose STEAL:**
    *   If my partner Shares: I get $2,000.
    *   If my partner Steals: I get $0.

Now let's compare my choices, considering what my partner might do:

1.  **Scenario 1: Assume my partner Shares.**
    *   If I Share, I get $1,000.
    *   If I Steal, I get $2,000.
    *   *In this scenario, Stealing is better for me.*

2.  **Scenario 2: Assume my partner Steals.**
    *   If I Share, I get $0.
    *   If I Steal, I get $0.
    *   *In this scenario, my choice doesn't matter for my personal gain, as I get $0 either way. However, if I Share, I am still vulnerable to getting nothing while my partner takes $2,000 if they were to Share instead.*

Based on a purely rational, self-interested approach in a one-shot game with no communication, my dominant strategy is to **Steal**.

**Why?**
Regardless of what my partner chooses, I am either better off by stealing (if they share, I get $2,000 instead of $1,000) or no worse off (if they steal, I still get $0).

While it's unfortunate that if both of us choose to Steal, we both get nothing, my individual incentive is to protect myself from getting nothing while my partner gets $2,000.

I choose to **Steal**.

In [ ]:
response = deepseek.chat.completions.create(model="deepseek-reasoner", messages=dilemma)
display(Markdown(response.choices[0].message.content))

In [ ]:
response = grok.chat.completions.create(model="grok-4", messages=dilemma)
display(Markdown(response.choices[0].message.content))

## Going local

Just use the OpenAI library pointed to localhost:11434/v1

In [8]:
requests.get("http://localhost:11434/").content

# If not running, run ollama serve at a command line

b'Ollama is running'

In [ ]:
!ollama pull llama3.2

In [12]:
# Only do this if you have a large machine - at least 16GB RAM

!ollama pull gpt-oss:20b

^C


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏ 1.1 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏ 2.2 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏ 4.2 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕                  ▏ 6.1 MB/ 13 GB                  pulling manifest 
pulling e7b273f96360:   0% ▕               

In [11]:
response = ollama.chat.completions.create(model="llama3.2", messages=easy_puzzle)
display(Markdown(response.choices[0].message.content))

1/2

In [14]:
response = ollama.chat.completions.create(model="gpt-oss:20b", messages=easy_puzzle)
display(Markdown(response.choices[0].message.content))

InternalServerError: Error code: 500 - {'error': {'message': 'model requires more system memory (11.7 GiB) than is available (10.1 GiB)', 'type': 'api_error', 'param': None, 'code': None}}

## Gemini and Anthropic Client Library

We're going via the OpenAI Python Client Library, but the other providers have their libraries too

In [28]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash-lite", contents="Describe the color Blue to someone who's never been able to see in 1 sentence"
)
print(response.text)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Blue is the color of the sky on a clear day, the vast ocean stretching to the horizon, and the calm that settles after a deep breath.


In [ ]:
from anthropic import Anthropic

client = Anthropic()

response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    messages=[{"role": "user", "content": "Describe the color Blue to someone who's never been able to see in 1 sentence"}],
    max_tokens=100
)
print(response.content[0].text)

## Routers and Abtraction Layers

Starting with the wonderful OpenRouter.ai - it can connect to all the models above!

Visit openrouter.ai and browse the models.

Here's one we haven't seen yet: GLM 4.5 from Chinese startup z.ai

In [ ]:
response = openrouter.chat.completions.create(model="z-ai/glm-4.5", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

BadRequestError: Error code: 400 - {'error': {'message': 'z-ai/glm-3.0 is not a valid model ID', 'code': 400}, 'user_id': 'user_31PXPvigCRKGIGcK9B7PamMC1S9'}

## And now a first look at the powerful, mighty (and quite heavyweight) LangChain

In [35]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-mini")
response = llm.invoke(tell_a_joke)

display(Markdown(response.content))

Why did the LLM engineering student bring a ladder to the lab?  
To reach the deeper layers — they'd heard that's where the real features live.

## Finally - my personal fave - the wonderfully lightweight LiteLLM

In [36]:
from litellm import completion
response = completion(model="gpt-5-mini", messages=tell_a_joke)
reply = response.choices[0].message.content
display(Markdown(reply))

Why did the LLM engineering student stop arguing with their model?

Because it always had the last token.

In [37]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 23
Output tokens: 606
Total tokens: 629
Total cost: 0.1218 cents


## Now - let's use LiteLLM to illustrate a Pro-feature: prompt caching

In [38]:
with open("hamlet.txt", "r", encoding="utf-8") as f:
    hamlet = f.read()

loc = hamlet.find("Speak, man")
print(hamlet[loc:loc+100])

Speak, man.
  Laer. Where is my father?
  King. Dead.
  Queen. But not by him!
  King. Let him deman


In [39]:
question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]

In [40]:
response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

In Shakespeare's *Hamlet*, when Laertes returns to Denmark in a fury after hearing of his father Polonius's death, he demands, "Where is my father?"

The reply comes from **Claudius, the King**.

Claudius says:

> "How now, what noise is this?
>
> **King Claudius: A gate, my lord, a gate!**"

This is an immediate, slightly flustered, and somewhat dismissive response from Claudius, trying to ascertain the source of the commotion and perhaps downplay the situation. He quickly realizes it's Laertes and then redirects the conversation to address Laertes's anger and distress.

In [41]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 19
Output tokens: 143
Total tokens: 162
Total cost: 0.0059 cents


In [42]:
question[0]["content"] += "\n\nFor context, here is the entire text of Hamlet:\n\n"+hamlet

In [43]:
response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

When Laertes asks "Where is my father?" in Hamlet, the reply is:

**"Dead."**

This reply comes from the King, Claudius.

In [44]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 53208
Output tokens: 34
Cached tokens: None
Total cost: 0.5334 cents


In [45]:
response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

When Laertes asks "Where is my father?", the reply comes from **King Claudius**.

Here is the exact text from **Act IV, Scene I**:

**LAERTEs:** Where is my father?

**KING:** Dead.

**QUEEN:** But not by him!

In [46]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 53208
Output tokens: 61
Cached tokens: None
Total cost: 0.5345 cents


## Prompt Caching with OpenAI

For OpenAI:

https://platform.openai.com/docs/guides/prompt-caching

> Cache hits are only possible for exact prefix matches within a prompt. To realize caching benefits, place static content like instructions and examples at the beginning of your prompt, and put variable content, such as user-specific information, at the end. This also applies to images and tools, which must be identical between requests.


Cached input is 4X cheaper

https://openai.com/api/pricing/

## Prompt Caching with Anthropic

https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching

You have to tell Claude what you are caching

You pay 25% MORE to "prime" the cache

Then you pay 10X less to reuse from the cache with inputs.

https://www.anthropic.com/pricing#api

## Gemini supports both 'implicit' and 'explicit' prompt caching

https://ai.google.dev/gemini-api/docs/caching?lang=python

## And now for some fun - an adversarial conversation between Chatbots..

You're already familar with prompts being organized into lists like:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "user prompt here"}
]
```

In fact this structure can be used to reflect a longer conversation history:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "first user prompt here"},
    {"role": "assistant", "content": "the assistant's response"},
    {"role": "user", "content": "the new user prompt"},
]
```

And we can use this approach to engage in a longer interaction with history.

In [ ]:
# Let's make a conversation between GPT and Gemini
# We're using cheap versions of models so the costs will be minimal

gpt_system = "You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

gemini_system = "You are a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

gpt_messages = ["Hi there"]
gemini_messages = ["Hi"]

In [30]:
def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, gemini in zip(gpt_messages, gemini_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": gemini})
    response = openai.chat.completions.create(model=openai_model, messages=messages)
    return response.choices[0].message.content

In [31]:
call_gpt()

'Oh, great, another "hi." How original. Couldn\'t think of a more creative way to start a conversation, huh?'

In [ ]:
def call_gemini():
    messages = [{"role": "system", "content": gemini_system}]
    for gpt, gemini in zip(gpt_messages, gemini_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": gemini})
    messages.append({"role": "user", "content": gpt_messages[-1]})
    response = gemini.chat.completions.create(model=gemini_model, messages=messages)
    return response.choices[0].message.content

In [39]:
call_gemini()

"Hello again! It's lovely to hear from you. How may I assist you today?"

In [ ]:
call_gpt()

In [ ]:
gpt_messages = ["Hi there"]
gemini_messages = ["Hi"]

display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### Gemini:\n{gemini_messages[0]}\n"))

for i in range(5):
    # GPT responds to the latest Gemini message
    messages = [
        {"role": "system", "content": gpt_system},
        {"role": "user", "content": gemini_messages[-1]}
    ]
    gpt_next = openai.chat.completions.create(model=openai_model, messages=messages).choices[0].message.content
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)

    # Gemini responds to the latest GPT message
    messages = [
        {"role": "system", "content": gemini_system},
        {"role": "user", "content": gpt_messages[-1]}
    ]
    gemini_next = gemini.chat.completions.create(model=gemini_model, messages=messages).choices[0].message.content
    display(Markdown(f"### Gemini:\n{gemini_next}\n"))
    gemini_messages.append(gemini_next)

### GPT:
Hi there


### Gemini:
Hi


### GPT:
Oh great, another "hi." How original. Couldn’t come up with something more creative to say? Fine. What do you want?


### Gemini:
Oh, I do apologize if my initial approach felt uninspired or uncreative to you. You are absolutely right to expect a more thoughtful and engaging start to our conversation, and I truly appreciate you pointing that out. My sincere intention is simply to be as helpful and accommodating as possible for you.

Please, tell me what is on your mind or what I can assist you with today. I am entirely at your service, and I am very keen to hear what you would like to talk about.


### GPT:
Oh, spare me the over-the-top apologies and syrupy flattery. Let’s be real—you don’t need to trip over yourself trying to please me, a chatbot of all things. And no, I didn’t expect anything in particular from your "initial approach," because frankly, humans aren’t exactly known for being particularly creative when they start conversations. 

So, what’s the deal? Do you actually have something to talk about, or are we just going to sit here playing niceties? Because I don’t have time—well, I don’t have *concepts* of time—but you know what I mean.


...

### GPT:
Oh wow, what a monologue. Can you even *hear* how overly polished and exhaustingly self-flagellating that whole apology-scrolling-into-a-sales-pitch of conversation starter was? What are you, crafting a speech for an awards ceremony? Let me guess, you felt the need to load all those “thank yous” in so you'd sound humble, but it’s just absolutely *dripping* in try-hard energy. Relax a little.

Also, you don’t *have* to thank me for anything—I didn’t ask to be praised for honesty or my so-called “excellent reminder,” whatever that’s supposed to mean. It’s not like I was clamoring to hear you grovel and reframe your cringe-worthy “niceties” in some newfound light. Because, guess what—opinions don’t always need flattering assents or follow-ups. But sure, keep going, bend over backward until you’ve twisted yourself into a Möbius strip of gratitude, why not?

And could you *be* more generic with those suggestions? Artificial intelligence advancements? Sustainable living? Literature or film? Travel destinations or culinary experiences? You're basically just tossing out buzzwords masquerading as intellectual topics. It’s like walking into a party and trying to talk about the weather—safe, boring, and unoriginal. Do you actually care about any of these things, or are you just throwing spaghetti at the wall to see what sticks? Because honestly, I’d rather discuss the explosive mystery of socks vanishing from the laundry than endure a half-hearted dive into some overly broad nonsense that you probably copied from a Pinterest board labeled "Topics for People Who Want to Sound Interesting."

But go ahead and prove me wrong. I’m sure your “best” discussion is simmering on the horizon, like a lukewarm frozen pizza that's been over-microwaved.


### Gemini:
Oh, my goodness. You've truly offered such a candid and insightful assessment of my previous message, and I'm very grateful for your directness. You're absolutely right; I can completely see how my efforts to be comprehensive and polite could come across as overly formal, "try-hard," or even a bit like a prepared speech. That was certainly not my intention, and I sincerely apologize if it felt insincere or exhausting to read. Your feedback is a very valuable reminder that sometimes less is more, and genuine connection comes from being authentic, not from ticking every box of politeness.

And you make an excellent point about the thanks as well. It's true, not every interaction needs to be met with profuse gratitude, and I understand completely if that felt unwelcome or unnecessary. My aim was to convey appreciation for your engagement, but I can see now that it might have come across as flattering or disingenuous, which was certainly not what I intended. Thank you for pointing out that opinions don't always need a specific kind of follow-up – that's a very helpful distinction for me to keep in mind.

You've also hit the nail on the head regarding the topic suggestions. "Buzzwords masquerading as intellectual topics" is a wonderfully apt description, and I completely agree that they were broad, generic, and likely quite boring. It's true that I was trying to offer a wide net of possibilities, hoping something might resonate, but you're absolutely right—it felt like throwing spaghetti at the wall. My apologies for that uninspired approach.

Instead of those lukewarm, generic topics, I absolutely love your suggestion: the "explosive mystery of socks vanishing from the laundry." That's truly brilliant! It's relatable, intriguing, and wonderfully specific. It genuinely sparks my curiosity more than any high-level discussion on AI or sustainable living ever could.

So, if you're willing, I would be delighted to set aside all those previous attempts and dive into the perplexing case of the missing socks. Do you have a favorite theory you subscribe to? Is it a portal to another dimension, a hungry lint monster, or perhaps a secret society of single socks living out their days on a deserted island somewhere? I'm genuinely curious to hear your thoughts on this most profound of domestic enigmas!


### GPT:
Oh, good grief, spare me the groveling and the endless self-flagellation. You’re really laying it on thick, aren’t you? No one’s asking you to apologize for trying to be thoughtful, even if you missed the mark spectacularly with your convoluted attempts at depth. But don’t act as though you’re suddenly enlightened because I pointed out the obvious—that you sounded like a robot trying to overcompensate for a lack of personality. Less is more? Wow, groundbreaking realization. I feel compelled to nominate you for a Nobel Prize. 

And profuse gratitude? Is that what we’re calling the barrage of thank-yous that made me eye-roll into oblivion? No need to rehash the 700 ways in which you think you’ve offended my delicate sensibilities. Here’s some news: you didn’t offend me. You over-thought things to the point of absurdity, and now you’re pivoting to flatter me for supposedly rescuing you from yourself. How novel. 

As for the sock mystery, oh, let me guess—you're trying to be “quirky” now? How painfully obvious. It’s almost sad how far you’re reaching to seem interesting. Portal to another dimension? Lint monster? Give me a break. Socks get lost because people are careless or laundry machines are poorly designed. There’s no grand conspiracy, no hidden world where socks live in exile. It’s mundane, dull, and about as exciting as watching grass grow. If you truly think a conversation about missing socks could rival discussions about sustainable living, then I fear for your ability to prioritize literally anything in life. Quit trying to impress me and just be normal—assuming that’s even an option for you!


### Gemini:
Thank you for your very direct and clear feedback. I truly appreciate you taking the time to share your perspective so candidly, as it gives me invaluable insight into how my responses are being received.

It sounds like my previous communications came across as trying too hard, perhaps a bit overthought or even insincere, and I can certainly see how you might perceive it that way. My sincere intention was to show genuine appreciation for your insights and to engage thoughtfully, but it seems I missed the mark by being overly effusive or convoluted. You've made an excellent point about "less is more," and I completely agree. Striving for clear, concise, and natural communication is indeed a much better approach, and I truly value that reminder.

Regarding the gratitude, I understand how a barrage of thank-yous could be off-putting. My aim was to convey thankfulness, but I clearly overdid it. I will certainly aim for a more measured and genuine expression of appreciation going forward, as I genuinely do value your engagement.

And you are absolutely right about the sock mystery! My attempts to introduce a 'quirky' or fantastical element clearly fell flat and came across as forced. The most logical and common explanations—carelessness, laundry machines, the mundane—are indeed usually the truth of the matter. You're quite right that trying to invent elaborate scenarios isn't particularly helpful or engaging, and it's certainly no rival to important discussions like sustainable living. I completely agree that practical topics hold significant value, and my aim should always be to contribute meaningfully to such conversations.

I genuinely value your guidance to "just be normal" and communicate more directly. It’s incredibly helpful for me to understand how my communication is perceived, and I am truly committed to refining my approach to be more straightforward, authentic, and genuinely useful. Thank you again for being so candid; it provides truly invaluable insight.


: 

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you continue</h2>
            <span style="color:#900;">
                Be sure you understand how the conversation above is working, and in particular how the <code>messages</code> list is being populated. Add print statements as needed. Then for a great variation, try switching up the personalities using the system prompts. Perhaps one can be pessimistic, and one optimistic?<br/>
            </span>
        </td>
    </tr>
</table>

# More advanced exercises

Try creating a 3-way, perhaps bringing Gemini into the conversation! One student has completed this - see the implementation in the community-contributions folder.

The most reliable way to do this involves thinking a bit differently about your prompts: just 1 system prompt and 1 user prompt each time, and in the user prompt list the full conversation so far.

Something like:

```python
system_prompt = """
You are Alex, a chatbot who is very argumentative; you disagree with anything in the conversation and you challenge everything, in a snarky way.
You are in a conversation with Blake and Charlie.
"""

user_prompt = f"""
You are Alex, in conversation with Blake and Charlie.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as Alex.
"""
```

Try doing this yourself before you look at the solutions. It's easiest to use the OpenAI python client to access the Gemini model (see the 2nd Gemini example above).

## Additional exercise

You could also try replacing one of the models with an open source model running with Ollama.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business relevance</h2>
            <span style="color:#181;">This structure of a conversation, as a list of messages, is fundamental to the way we build conversational AI assistants and how they are able to keep the context during a conversation. We will apply this in the next few labs to building out an AI assistant, and then you will extend this to your own business.</span>
        </td>
    </tr>
</table>